[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PietropaoloFrisoni/qrisp-community-day-2026/blob/main/JaspExecution.ipynb)

# Execution of a program in JAX and Jasp

We know how to get a `jaxpr` out of a function (and a `jaspr` from a qrisp quantum program). 

In this notebook we see what `jax.jit` actually does with it: how it turns that jaxpr into compiled code that runs fast.

Most of the concepts in this notebook build on what we've already seen in the previous notebook. Here we just see the full pipeline instead of the first step (tracing).

## Setup

In [ ]:
!pip install qrisp

In [61]:
import jax
import jax.numpy as jnp
import time

## Applying `jax.jit` for the first time

The most natural way to perform Just-In-Time compilation with JAX is through `jax.jit`. Let's see it in action.

We define a plain `f` Python function, which does some repeated array math.

In [62]:
def f(x):
    for _ in range(50):
        x = 0.25 * x**3 + 0.75 * x
    return jnp.sum(x)

At first, we time it as-is, so we have a plain baseline to compare every compiled version against later.

In [63]:
x = jnp.ones((1000, 1000))

start = time.perf_counter()
for _ in range(50):
    _ = f(x)
print(f"Without (jit): {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call")

Without (jit): 297.68 ms per call


Let's see what changes once `jax.jit` gets involved.

`jax.jit(f)` doesn't compile anything by itself. It just wraps `f` so the *first* call traces and compiles it, and later calls (with matching input shapes/dtypes) **reuse** that compiled version.

Watch the timings below shift as we go from "no jit" to "first jit call" to "cached jit call."

In [64]:
f_jit = jax.jit(f)

Every timed call below ends in `.block_until_ready()`. This is necessary because JAX dispatches work asynchronously, so without it we'd be timing how fast Python hands off the call, not how long the computation actually takes (more on exactly why in the "Asynchronous Dispatch" section).

In [65]:
# First call (tracing + compilation)
start = time.perf_counter()
_ = f_jit(x).block_until_ready()
elapsed = (time.perf_counter() - start) * 1000
print(f"First call (trace + compile): {elapsed:.2f} ms")

First call (trace + compile): 363.27 ms


In [66]:
# Subsequent calls (use the cached compiled function)
start = time.perf_counter()
for _ in range(50):
    _ = f_jit(x).block_until_ready()
print(f"Subsequent calls: {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call")

Subsequent calls: 20.48 ms per call


Observations:

- If we maintain the same shape and type of dynamic arguments, then the jaxpr is the same and, therefore, JAX does not have to re-compile our function: we have true just-in-time compilation!

- In practice, JAX doesn't retrace and compare jaxprs to decide this: it keys its cache on the input shapes/dtypes directly, so a matching signature skips tracing entirely and reuses the cached compiled version; a new signature forces a fresh trace (and, since the computation may now look different, possibly a different jaxpr).

- The first call already blends one real compilation together with 49 cache hits in the same timed loop. That's why its average doesn't look as expensive as a raw compile would.

We can also write the same function using `@jax.jit` as a decorator instead of calling `jax.jit(f)` directly:

In [67]:
@jax.jit
def f_decorated(x):
    for _ in range(50):
        x = 0.25 * x**3 + 0.75 * x
    return jnp.sum(x)


_ = f_decorated(x).block_until_ready()

## Asynchronous Dispatch and `block_until_ready` (bonus)

Calling a jitted function hands work off to the accelerator and returns right away. It doesn't block until the result is actually computed.

The first call below also doubles as a warm-up (so the timings that follow aren't paying a compilation cost); `block_until_ready()` is what forces Python to actually wait for a result.

In [68]:
@jax.jit
def heavy(a):
    return a @ a


a = jnp.ones((2000, 2000))
# this call triggers trace + compile.
# That is, a warm-up before we time dispatch vs. compute separately
heavy(a).block_until_ready()

Array([[2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       ...,
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.]], dtype=float64)

In [69]:
start = time.perf_counter()
result = heavy(a)
print(f"Call returned after: {(time.perf_counter() - start) * 1000:.3f} ms")

Call returned after: 2.757 ms


In [70]:
start = time.perf_counter()
heavy(a).block_until_ready()
print(f"Total time (dispatch + compute): {(time.perf_counter() - start) * 1000:.3f} ms")

Total time (dispatch + compute): 510.844 ms


Observations:

- The bare call above returns in a fraction of a millisecond. This is dispatch overhead, not the matmul itself: the accelerator keeps working after Python already moved on.

- Adding `block_until_ready()` exposes the real total time, dispatch plus compute together. This is the async gap in action.

## The Compilation Cache (bonus)

The jit cache is keyed on the *shape and dtype* of the inputs, not their concrete values. Same shape means a cache hit, a new shape means JAX has never seen this "version" of the function before.

This is exactly the same behaviour we have already seen for the `jaxpr`, which is conceptually the first IR of the entire Just-In-Time compilation process.
The difference now is that `jit` doesn't stop at the `jaxpr`: it keeps compiling all the way down to machine code. We open up exactly what happens in between later in this notebook, in "Inspecting What Gets Compiled".

The three calls below go compile → cache-hit → recompile, back to back.

In [71]:
@jax.jit
def g(x):
    return x * 2.0


# Compile for shape (10,)
start = time.perf_counter()
_ = g(jnp.ones(10)).block_until_ready()
print(f"Compiled for shape (10,): {(time.perf_counter() - start) * 1000:.2f} ms")

# Cache hit (same shape)
start = time.perf_counter()
_ = g(jnp.ones(10)).block_until_ready()
print(f"Compiled for shape (10,): {(time.perf_counter() - start) * 1000:.2f} ms")

# Recompile (new shape)
start = time.perf_counter()
_ = g(jnp.ones(20)).block_until_ready()
print(f"Recompile for shape (20,): {(time.perf_counter() - start) * 1000:.2f} ms")

Compiled for shape (10,): 64.20 ms
Compiled for shape (10,): 1.65 ms
Recompile for shape (20,): 65.55 ms


Same shape → fast (cache hit). 

new shape → slow again (fresh trace + compile), because the cache has no way to reuse a compiled program for an input signature it hasn't seen.

## Inspecting What Gets Compiled (bonus)

`jit` builds a compiled program behind the scenes, and `.lower(x)` lets you grab a handle on it without running it, and print it out at different levels of representation.

We have to make a distinction here: StableHLO is a portable, versioned dialect meant to be exchanged across tools and JAX/XLA versions (hence "stable"), while HLO is XLA's own internal representation, free to change under the hood. Below is the same function shown as both.

In [72]:
@jax.jit
def h(x):
    return jnp.sin(x) + jnp.cos(x)

In [73]:
x = jnp.ones(4)

In [74]:
lowered = h.lower(x)
print(lowered.as_text(dialect="stablehlo"))

module @jit_h attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4xf64>) -> (tensor<4xf64> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<4xf64>
    %1 = stablehlo.cosine %arg0 : tensor<4xf64>
    %2 = stablehlo.add %0, %1 : tensor<4xf64>
    return %2 : tensor<4xf64>
  }
}



In [75]:
print(lowered.as_text(dialect="hlo"))

HloModule jit_h, entry_computation_layout={(f64[4]{0})->f64[4]{0}}

ENTRY main.5 {
  x.1 = f64[4]{0} parameter(0)
  sin.2 = f64[4]{0} sine(x.1)
  cos.3 = f64[4]{0} cosine(x.1)
  ROOT add.4 = f64[4]{0} add(sin.2, cos.3)
}




Same three operations either way. Notice HLO is basically the StableHLO version respelled in XLA's native syntax, not a different computation.

## `disable_jit` for Debugging (bonus)

`jax.disable_jit()` turns every `jit` inside the block into a no-op, so code runs as plain, eager Python: handy for stepping through with a debugger or trusting every `print` you see.

Compare how many times "Tracing!" prints with and without it:

In [76]:
@jax.jit
def f(x):
    print("Tracing!")
    return x * 2.0

In [77]:
print("=== With jit ===")
_ = f(jnp.array(1.0))
_ = f(jnp.array(2.0))

=== With jit ===
Tracing!


In [78]:
print("=== With disable_jit ===")
with jax.disable_jit():
    _ = f(jnp.array(1.0))
    _ = f(jnp.array(2.0))

=== With disable_jit ===
Tracing!
Tracing!


Observations:

- With `jit`: tracing (and the print) happens once. The second call is a cache hit and stays silent.

- With `disable_jit`: there's no cache at all, so every call re-runs the Python body from scratch, printing every time.

## `jit` Itself Is a Primitive (bonus)

As we saw with jaxprs, every JAX operation becomes a primitive in the trace, and `jit` is no exception.

When a jitted function is called from inside another traced function, it isn't inlined. It shows up as its own `jit` primitive wrapping a nested jaxpr.

In [79]:
def outer(arg):
    @jax.jit
    def inner(x):
        return x + arg * jnp.ones(1)

    return arg + inner(arg - 2.0)

In [80]:
print(jax.make_jaxpr(outer)(1.0))

{ lambda ; a:f64[]. let
    b:f64[] = sub a 2.0:f64[]
    c:f64[1] = jit[
      name=inner
      jaxpr={ lambda ; a:f64[] b:f64[]. let
          d:f64[1] = broadcast_in_dim[
            broadcast_dimensions=()
            shape=(1,)
            sharding=None
          ] 1.0:f64[]
          e:f64[] = convert_element_type[new_dtype=float64 weak_type=False] a
          f:f64[1] = mul e d
          g:f64[] = convert_element_type[new_dtype=float64 weak_type=False] b
          c:f64[1] = add g f
        in (c,) }
    ] a b
    h:f64[] = convert_element_type[new_dtype=float64 weak_type=False] a
    i:f64[1] = add h c
  in (i,) }


See the `jit[name=inner, jaxpr={...}]` entry in the middle of the printout: that's `inner` kept as its own nested sub-program rather than flattened into `outer`'s jaxpr.

## `qache`: Jasp's Compilation Cache (bonus)

This is the mechanism qrisp's Jasp module reuses for caching.

In Jasp, the `@qache` decorator allows us to mark a function as 'reusable' (that is, the jasp expression of this function will be cached and reused in the next calls if the function is called with the same signature).

Using the qache decorator not only improves the compilation speed but also enables the compiler to speed up transformation processes.

Below, we create a simple function that is 'qached'. 
To simulate an expensive compilation task we insert a `time.sleep` command:

In [81]:
from qrisp import h, cx, measure, QuantumVariable, QuantumFloat, make_jaspr
from qrisp.jasp import qache


@qache
def qached_function(qv):
    h(qv[0])
    cx(qv[0], qv[1])
    res_bl = measure(qv[0])
    # Simulate demanding compilation procedure by calling
    time.sleep(1)
    return res_bl


def main():
    a = QuantumVariable(2)
    b = QuantumFloat(2)
    bl_0 = qached_function(a)
    bl_1 = qached_function(b)
    bl_2 = qached_function(a)
    bl_3 = qached_function(b)

    return bl_0, bl_1, bl_2, bl_3


# Measure the time required for tracing
start = time.perf_counter()
jaspr = make_jaspr(main)()
print(f"Tracing time: {(time.perf_counter() - start) * 1000:.2f} ms")

Tracing time: 2040.81 ms


Even though `qached_function` has been called 4 times, we only see a delay of 2 seconds. 

This is because the function has been called with two different quantum types, implying it has been traced twice and recalled from the cache twice. 

We can see all of this by looking at the `jaspr`:

In [82]:
print(jaspr)

let qached_function = { lambda ; a:QubitArray b:QuantumState. let
    c:Qubit = jasp.get_qubit a 0:i64[]
    d:QuantumState = jasp.quantum_gate[gate=h] c b
    e:Qubit = jasp.get_qubit a 1:i64[]
    f:QuantumState = jasp.quantum_gate[gate=cx] c e d
    g:bool[] h:QuantumState = jasp.measure c f
  in (g, h) } in
let qached_function1 = { lambda ; i:QubitArray j:i64[] k:QuantumState. let
    l:Qubit = jasp.get_qubit i 0:i64[]
    m:QuantumState = jasp.quantum_gate[gate=h] l k
    n:Qubit = jasp.get_qubit i 1:i64[]
    o:QuantumState = jasp.quantum_gate[gate=cx] l n m
    p:bool[] q:QuantumState = jasp.measure l o
  in (p, q) } in
{ lambda ; r:QuantumState. let
    s:QubitArray t:QuantumState = jasp.create_qubits 2:i64[] r
    u:QubitArray v:QuantumState = jasp.create_qubits 2:i64[] t
    w:bool[] x:QuantumState = jit[name=qached_function jaxpr=qached_function] s v
    y:bool[] z:QuantumState = jit[name=qached_function jaxpr=qached_function1] u
      0:i64[] x
    ba:bool[] bb:QuantumState

We see three different function definitions in the `jaspr`:

- The first one describes `qached_function` called with a `QuantumVariable`. For this kind of signature only the `QubitArray` is required.

- The second one describes `qached_function` called with `QuantumFloat`. Additionally to the `QubitArray`, the `.exponent` attribute is also passed as a traced value (the extra `i64[]` argument).

- The third function definition is `main`, which calls the previously defined functions.

## Resource Estimation at Scale in qrisp

We've seen `jax.jit`'s cache decide, based only on shape and dtype, whether to reuse a compiled program or trace again. Now let's see the first way Jasp gets to reuse that exact same compiled pipeline.

Jasp also ships decorators that compile a function into a *resource estimator* instead of a simulator or a real quantum computer.

For example, `count_ops` traces the function once and returns a dictionary of gate counts, without ever running an actual quantum simulation.

This works for exactly the reason we saw with the shape/dtype-keyed `jit` cache above: if the size of a circuit is a *dynamic* argument (a traced value, not a Python `int` fixed ahead of time), tracing it once covers every possible size. Growing it doesn't trigger a new trace, only the classical gate-counting arithmetic has to scale.

In [83]:
from qrisp import count_ops, QuantumVariable, measure, h
from qrisp.jasp import jrange

# This is just a trick to count how many times the function is traced
trace_count = [0]


@count_ops(meas_behavior="0")
def main(n):
    trace_count[0] += 1
    qv = QuantumVariable(n)
    for i in jrange(n):
        h(qv[i])
    return measure(qv)


start = time.perf_counter()
result_small = main(5)
elapsed = (time.perf_counter() - start) * 1000
print(f"main(5): {elapsed:.1f} ms -> {result_small}")

start = time.perf_counter()
result_large = main(10_000_000)
elapsed = (time.perf_counter() - start) * 1000
print(
    f"main(10_000_000): {elapsed:.1f} ms -> total gates = {sum(result_large.values())}"
)

print(f"\nNumber of traces: {trace_count[0]}")

main(5): 164.1 ms -> {'h': 5, 'measure': 5}
main(10_000_000): 33.7 ms -> total gates = 20000000

Number of traces: 1


Observations:

- `main(5)` allocates a 5-qubit `QuantumVariable` and puts every qubit into superposition with `h`: `{'h': 5, 'measure': 5}`, exactly one Hadamard and one measurement per qubit.

- `main(10_000_000)` does the exact same thing on 10 million qubits (20 million gates total) and counting them still takes a few milliseconds.

- `trace_count` proves why: `n` is a dynamic argument, so growing it from 5 to 10,000,000 doesn't retrace at all. Only the classical gate-counting logic has to scale, not the number of times Python gets involved.

`count_ops` is not the only resource estimator: `num_qubits` and `depth` work the same way.

In [84]:
from qrisp.jasp import depth, num_qubits


def circuit(n):
    qv = QuantumVariable(n)
    for i in jrange(n):
        h(qv[i])
    return measure(qv)


print(f"num_qubits(10_000_000): {num_qubits(meas_behavior='0')(circuit)(10_000_000)}")
print(f"depth(1000): {depth(meas_behavior='0')(circuit)(1000)}")

num_qubits(10_000_000): {'total_allocated': 10000000, 'total_deallocated': 0, 'peak_allocations': 10000000, 'finally_allocated': 10000000}
depth(1000): 1


Observations:

- `num_qubits` returns how many qubits are allocated and deallocated during the program, and the peak number in use at the same time. Like `count_ops`, it handles 10 million qubits in a fraction of a second.

- The depth is 1 for any `n`: the Hadamards act on different qubits, so they all fit in a single layer (measurements are not counted).

- `depth` keeps one counter per qubit, in an array of fixed size (`max_qubits`, 1024 by default). That is why we use `n = 1000` here: it does not scale to millions of qubits like the other two.

If you want to see this at scale, please check out [*Using `BigInteger` to compile Shor's algorithm at 2048 bit*](https://www.qrisp.eu/general/tutorial/BigInteger.html) in the official qrisp documentation. It applies this exact `count_ops` technique to resource-estimate Shor's algorithm at a cryptographically relevant size.

## Boolean Simulation

If a whole circuit is built only out of classical gates (that is, gates such as X, CX, CCX, SWAP, etc. which only ever permute basis states), there is no need for a real quantum-state simulator at all. That is, every qubit is either `0` or `1`, always, and every gate is just an ordinary bit operation.

`boolean_simulation` is a decorator defined within the Jasp submodule which exploits exactly this. It rewrites the Jaspr into a version with no quantum primitives left at all (qubits become classical bits packed into an array and gates become bitwise operations on that array). And because it's now ordinary JAX code, it goes through the real `jax.jit` pipeline we have seen above.

The catch: it only works for circuits that never leave classical territory. Anything that creates superposition (`h`, `rx`, `ry`, ...) will raise an exception.

In [85]:
from qrisp import QuantumVariable, measure, x
from qrisp.jasp import boolean_simulation, jrange


@boolean_simulation
def main(n):
    qv = QuantumVariable(n)
    for i in jrange(n):
        x(qv[i])
    return measure(qv)

In [86]:
from qrisp import count_ops


# Same circuit body, but count_ops (not boolean_simulation) to see its actual size
@count_ops(meas_behavior="0")
def main_gate_count(n):
    qv = QuantumVariable(n)
    for i in jrange(n):
        x(qv[i])
    return measure(qv)


gate_counts = main_gate_count(1000)
print(f"Total gates in this circuit: {sum(gate_counts.values())}")

# Initial run (compilation)
start = time.perf_counter()
result = main(1000)
print(f"Result: {result}, Time: {(time.perf_counter() - start) * 1000:.2f} ms")

# Subsequent run (cache hit)
start = time.perf_counter()
result = main(1000)
print(f"Result: {result}, Time: {(time.perf_counter() - start) * 1000:.2f} ms")

Total gates in this circuit: 2000
Result: -1, Time: 252.84 ms
Result: -1, Time: 0.79 ms


Observations:

- `main(1000)` flips every one of its 1000 qubits with `x`, deterministically: the result is identical on both calls, purely classical logic, purely classical result.

- This example has 2000 gates (1000 `x` + 1000 `measure`), and yet both calls finish in a fraction of a second. We can test examples at scale, at native JAX speed, with none of the exponential cost of a real quantum-state simulation.

## Simulating a generic Jaspr

Any `jaspr` can be simulated by simply calling it like a function. Let's take a small example:

In [87]:
from qrisp import QuantumVariable, x, cx, h, measure, make_jaspr


def simple_circuit():
    qv = QuantumVariable(3)
    x(qv[0])
    cx(qv[0], qv[1])
    h(qv[2])
    return measure(qv)


jaspr = make_jaspr(simple_circuit)()
print(jaspr())

7                                                                               


Observations:

- Every call executes the program once, like a single shot on a real device. Qubits 0 and 1 are always `1` (`x` and `cx`), while `h` makes qubit 2 random, so we get either `3` (`0b011`) or `7` (`0b111`).

- Calling a jaspr runs qrisp's `simulate_jaspr`: it goes through the jaspr one equation at a time in Python, and applies every quantum primitive to qrisp's own statevector simulator. Measurements are sampled and collapse the state. Only purely classical `jit` sub-functions are compiled with `jax.jit`.

- Unlike `boolean_simulation`, this works for any jaspr: any gate, mid-circuit measurements and dynamic control flow.

- `jaspify(simple_circuit)()` does the same in one step: it builds the jaspr and calls it.

### Repeat Until Success in qrisp

Some procedures only succeed with a certain probability, and a measurement tells us whether they did. The `RUS` (repeat-until-success) decorator turns such a *trial function* into a loop that runs until the trial succeeds, so the number of iterations is only known at run time.

A trial function returns a success flag (a measurement result) as its first value, followed by the quantum variables it produced. When a trial fails, `RUS` resets and discards those variables and tries again. Local variables that are not returned have to be cleaned up by hand.

As a minimal example, the trial below applies `h` to a qubit and measures it: it succeeds when the measurement returns `True`, which happens with probability $1/2$.

In [88]:
from qrisp import QuantumVariable, h, measure
from qrisp.jasp import RUS, make_jaspr


def coin_flip():
    qv = QuantumVariable(1)
    h(qv[0])
    success = measure(qv[0])
    return success, qv


jaspr = make_jaspr(coin_flip)()
print(jaspr)

{ lambda ; a:QuantumState. let
    b:QubitArray c:QuantumState = jasp.create_qubits 1:i64[] a
    d:Qubit = jasp.get_qubit b 0:i64[]
    e:QuantumState = jasp.quantum_gate[gate=h] d c
    f:bool[] g:QuantumState = jasp.measure d e
  in (f, b, g) }


In [89]:
# The first return value is the success flag
print([bool(jaspr()[0]) for _ in range(5)])

[True, True, False, True, True]                                                 


With `RUS`, the same trial is repeated until `success` is `True`:

In [90]:
@RUS
def coin_flip():
    qv = QuantumVariable(1)
    h(qv[0])
    success = measure(qv[0])
    return success, qv


print(make_jaspr(coin_flip)())

let coin_flip = { lambda ; a:QuantumState. let
    b:QubitArray c:QuantumState = jasp.create_qubits 1:i64[] a
    d:Qubit = jasp.get_qubit b 0:i64[]
    e:QuantumState = jasp.quantum_gate[gate=h] d c
    f:bool[] g:QuantumState = jasp.measure d e
  in (f, b, g) } in
{ lambda ; h:QuantumState. let
    i:bool[] j:QubitArray k:QuantumState = jit[name=coin_flip jaxpr=coin_flip] h
    l:i32[] = convert_element_type[new_dtype=int32 weak_type=False] i
    _:bool[] m:QubitArray n:QuantumState = cond[
      branches=(
        { lambda ; o:bool[] p:QubitArray q:QuantumState. let
            r:bool[] s:QubitArray t:QuantumState = while[
              body_jaxpr={ lambda ; u:bool[] v:QubitArray w:QuantumState. let
                  x:QuantumState = jasp.reset v w
                  y:QuantumState = jasp.delete_qubits v x
                  z:bool[] ba:QubitArray bb:QuantumState = jit[
                    name=coin_flip
                    jaxpr=coin_flip
                  ] y
                in (z, 

`RUS` only rewrites the program, so we simulate it exactly as before, by calling the jaspr:

In [91]:
def main():
    qv = coin_flip()
    return measure(qv)


jaspr = make_jaspr(main)()
print(bool(jaspr()))

True                                                                            


Observations:

- `RUS` injects a `cond` and a `while` into the jaspr. `coin_flip` runs once, and the `cond` checks its success flag: if it is `True` (branch 1) nothing else happens, otherwise (branch 0) the `while` loop resets and deletes the failed qubit and calls `coin_flip` again, until the flag is `True`.

- A single trial succeeds with probability $1/2$, so on average it takes two trials, but the exact number is decided by measurements at run time.

- Every call returns `1`: within a single call, the trial is repeated until `success` is `True`, and the qubit stays in $|1\rangle$ after that last measurement. The execution path is the same as in the previous section: the `cond` and the `while` run in Python on qrisp's statevector simulator, and every trial samples a new measurement result. `RUS` needs nothing special from the simulator, it just builds the loop (and the cleanup) into the jaspr for us.

`RUS` is used in several quantum algorithms within qrisp. For example, the [HHL tutorial](https://www.qrisp.eu/general/tutorial/HHL.html) in the qrisp documentation uses it to solve systems of linear equations. Its trial function has the same structure as `coin_flip`, with a success flag that requires two measurements to return `0`.

## Compilation to QIR

So far every jaxpr and jaspr in this notebook has run through either JAX's own `jit`/XLA path or Jasp's own Python-level interpreter. There's a third option: compiling the whole jaspr, quantum part included, down to QIR via [Catalyst](https://docs.pennylane.ai/projects/catalyst/), Xanadu's JIT/AOT compiler for JAX and PennyLane programs. It is an optional extra, not installed by `qrisp` by default:

In [ ]:
!pip install qrisp[catalyst]

qrisp's `qjit` traces our function to a `jaspr`, translates every Jasp primitive into the compiler's own equivalent (`quantum_gate` → `qinst`, `measure` → `measure`, and so on), then compiles and runs the result. Just like `sin` lowers into `stablehlo.sine`, `qinst` itself lowers further into the `quantum.custom` ops shown below.

Its calling convention mirrors `jax.jit` exactly. The first call traces and compiles (now all the way to a runnable artifact), a later call with a matching signature is cached:

In [93]:
from qrisp import QuantumFloat, cx, h, measure
from qrisp.jasp import qjit


@qjit
def test_fun(i):
    qv = QuantumFloat(i)
    cx(qv[0], qv[1])
    h(qv[1])
    return measure(qv)


start = time.perf_counter()
result_first = test_fun(4)
elapsed = (time.perf_counter() - start) * 1000
print(f"First call (compile + run): {elapsed:.1f} ms -> {result_first}")

start = time.perf_counter()
result_cached = test_fun(4)
print(f"Cached call: {(time.perf_counter() - start) * 1000:.1f} ms -> {result_cached}")

First call (compile + run): 369.0 ms -> 0.0
Cached call: 44.7 ms -> 2.0


Observations:

- This has the same shape as every other cache-hit demo in this notebook: a slow first call (trace + compile + run), a fast cached call. Just with a compiler for the quantum part behind it instead of XLA.

- By default the compiled program runs on a statevector simulator (`lightning.qubit`), one shot per call: `test_fun` measures a quantum state, so the two calls can return different numbers even for the same `i=4`.

We can see the translation directly by printing the MLIR the `jaspr` is translated into:

In [94]:
from qrisp import QuantumFloat, cx, h, measure
from qrisp.jasp import make_jaspr


def example_function(i):
    qv = QuantumFloat(i)
    cx(qv[0], qv[1])
    h(qv[1])
    meas_res = measure(qv)
    return meas_res + 1


jaspr = make_jaspr(example_function)(2)
print(jaspr.to_catalyst_mlir())

module @jaspr_function {
  func.func public @jit_jaspr_function(%arg0: tensor<i64>) -> tensor<f64> attributes {llvm.emit_c_interface} {
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f64>
    %c = stablehlo.constant dense<1024> : tensor<i64>
    %c_0 = stablehlo.constant dense<false> : tensor<i1>
    %c_1 = stablehlo.constant dense<1> : tensor<i64>
    %c_2 = stablehlo.constant dense<30> : tensor<i64>
    %c_3 = stablehlo.constant dense<[30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]> : tensor<30xi64>
    %c_4 = stablehlo.constant dense<0> : tensor<i64>
    %c0_i64 = arith.constant 0 : i64
    quantum.device shots(%c0_i64) [".../pennylane_lightning/liblightning_qubit_catalyst.so", "LightningSimulator", "{'mcmc': False, 'num_burnin': 0, 'kernel_name': None}"] {auto_qubit_management}
    %0 = quantum.alloc( 20) : !quantum.reg
    %1 = stablehlo.broadcast_in_dim %c_4, dims = [] : (tensor<i64>) -> tensor<1024xi64>


Observations:

- `quantum.device`, `quantum.alloc`, `quantum.custom "cx"` and so on belong to the compiler's own `quantum` dialect: a completely different spelling from the `jasp.*` primitives used everywhere else in this notebook.

- The classical structure is not reinvented: the `stablehlo.constant`/`stablehlo.while` machinery is still plain JAX StableHLO. Only the quantum-specific parts get translated.

The `to_qir()` method goes one level lower than the MLIR above: it continues the same compilation and returns QIR, that is, LLVM IR in which every quantum operation has become a call to a runtime function.

In [95]:
qir_text = jaspr.to_qir()
print(f"Total length: {len(qir_text)} characters\n")
print(qir_text[:216])

Total length: 10827 characters

; ModuleID = 'LLVMDialectModule'
source_filename = "LLVMDialectModule"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"



Observations:

- Genuine LLVM IR syntax (`ModuleID`, `source_filename`, `target datalayout`, `target triple`): nothing is left of the `quantum.*` dialect at this stage, and nothing of `jasp.*` either. This is the actual compilation target, not a dialect anymore.

- Same `jaspr`, same compilation as the MLIR example above, just a later stage of it.

## One Program, Many Evaluation Modes

A jaspr is a program, and Jasp can hand it to different *interpreters*: a shot-by-shot simulator, a Clifford simulator, a classical bit simulator, or a resource estimator. Each one gives the same program a different meaning, trading generality for scale. 

Under the hood there are two routes: the shot-by-shot and Clifford simulators replay the jaspr equation by equation in Python, while the classical bit simulator and the resource estimator first replace every quantum operation with classical code, so that the whole program can be compiled with `jax.jit`.

To see them all at work on one example, here is a program that is dynamic in every sense of this notebook: one round of the bit-flip **repetition code**, the simplest quantum error correction code.

- `num_data_qubits` data qubits encode a logical bit, with `num_data_qubits` a traced value (dynamic size).
- A bit-flip error is injected on one data qubit.
- `num_data_qubits - 1` ancillas measure the parities of neighbouring data qubits: the *syndrome*.
- A decoder turns the syndrome into a correction, applied in real time (dynamic control flow).

In [96]:
from qrisp import QuantumVariable, x, cx, measure, control
from qrisp.jasp import jrange


def error_signature(qubit, num_checks):
    # Syndrome of a bit-flip on `qubit`: the checks on its left and right fire
    # (only one of them for the first and the last data qubit)
    left_and_right = (1 << qubit) | ((1 << qubit) >> 1)
    return left_and_right & ((1 << num_checks) - 1)


def repetition_code(num_data_qubits, logical_bit, error_position):
    num_checks = num_data_qubits - 1
    data = QuantumVariable(num_data_qubits)
    ancillas = QuantumVariable(num_checks)

    # Encode the logical bit: |0...0> or |1...1>
    with control(logical_bit):
        x(data)

    # Inject a bit-flip error on one data qubit
    x(data[error_position])

    # Syndrome extraction: ancilla i stores the parity of data qubits i and i + 1
    cx(data[:-1], ancillas)
    cx(data[1:], ancillas)
    syndrome = measure(ancillas)

    # Real-time decoding: flip the data qubit whose error signature matches the syndrome
    for qubit in jrange(num_data_qubits):
        with control(syndrome == error_signature(qubit, num_checks)):
            x(data[qubit])

    return measure(data)

First, the three simulators. `stimulate` uses Stim, an optional dependency:

In [ ]:
!pip install stim

In [98]:
from qrisp.jasp import jaspify, stimulate, boolean_simulation

num_data_qubits, logical_bit, error_position = 5, True, 2
args = (num_data_qubits, logical_bit, error_position)

print(f"jaspify:            {jaspify(repetition_code)(*args)}")
print(f"stimulate:          {stimulate(repetition_code)(*args)}")
print(f"boolean_simulation: {boolean_simulation(repetition_code)(*args)}")

jaspify:            31                                                          
stimulate:          31
boolean_simulation: 31


All three recover the logical `1` (`31` is `11111`) despite the error. They differ in what they accept and how far they scale:

- `jaspify` accepts any gate set, but a general quantum state costs memory exponential in the number of qubits.
- `stimulate` accepts only Clifford gates (plus measurements and real-time control), which is exactly what error correction circuits use.
- `boolean_simulation` accepts only classical-reversible gates (X, CX, CCX, ...) and compiles the program with `jax.jit`.

Our circuit qualifies for all three, so we can push it to a size where general state simulation is hopeless:

In [99]:
num_data_qubits, error_position = 60, 37  # 60 data qubits + 59 ancillas

for name, simulator in [
    ("stimulate", stimulate),
    ("boolean_simulation", boolean_simulation),
]:
    start = time.perf_counter()
    result = simulator(repetition_code)(num_data_qubits, True, error_position)
    elapsed = (time.perf_counter() - start) * 1000
    recovered = result == 2**num_data_qubits - 1
    print(f"{name}: logical 1 recovered = {recovered} ({elapsed:.0f} ms)")

stimulate: logical 1 recovered = True (199 ms)
boolean_simulation: logical 1 recovered = True (1949 ms)


Finally, the resource estimators. `count_ops`, `depth` and `num_qubits` reinterpret the same jaspr as a program that counts: every gate becomes a counter update, and the resulting classical program is compiled with `jax.jit`. For a dynamic program they need an assumption about the measurement results, set by `meas_behavior`:

In [100]:
from qrisp.jasp import count_ops, depth, num_qubits

num_data_qubits, logical_bit, error_position = 5, True, 2
args = (num_data_qubits, logical_bit, error_position)

print(f"count_ops (all 0): {count_ops(meas_behavior='0')(repetition_code)(*args)}")
print(f"count_ops (sim):   {count_ops(meas_behavior='sim')(repetition_code)(*args)}")
print(f"depth:             {depth(meas_behavior='0')(repetition_code)(*args)}")
print(f"num_qubits:        {num_qubits(meas_behavior='0')(repetition_code)(*args)}")

count_ops (all 0): {'x': 6, 'cx': 8, 'measure': 9}
count_ops (sim):   {'x': 7, 'cx': 8, 'measure': 9}                              
depth:             4
num_qubits:        {'total_allocated': 9, 'total_deallocated': 0, 'peak_allocations': 9, 'finally_allocated': 9}


Observations:

- With `meas_behavior='0'` every measurement is assumed to return 0, so no error is ever detected and the correction is never counted: 6 `x` gates (5 for the encoding, 1 for the injected error).

- With `meas_behavior='sim'` the program is simulated to obtain actual measurement results, the decoder fires, and the correction shows up as a 7th `x` gate. For a dynamic program, the resource counts depend on the measurement results.

- `num_qubits` reports 9 qubits: 5 data qubits and 4 ancillas.

- This is the same idea behind Qrisp's 2048-bit Shor resource estimates: one traced program, reinterpreted by different interpreters, including one that only counts.

- `measure` returns a 64-bit integer here, so this version of the decoder works up to `num_data_qubits = 63`.

## Keep Exploring

We've only had time for a first look at tracing, real-time control flow, repeat-until-success loops, and the main evaluation modes. Jasp has quite a bit more. For example, a jaspr can also be unrolled into a static circuit for hardware backends, OpenQASM or Stim (`to_qc`, `to_qasm`, `extract_stim`, `backend_sampler`), and real-time classical logic can go as far as running neural network inference inside a coherence window.

For the full picture, see the [Jasp documentation](https://www.qrisp.eu/general/tutorial/Jasp.html). Hopefully this tutorial gave you a solid enough high-level understanding to keep exploring from here on your own.

## ... what comes next?

If you want to go further, the `BonusMaterial/` folder has three optional notebooks: 

- `PrimitivesInJax.ipynb` (build a custom JAX primitive from scratch, with rules for implementation, abstract evaluation, lowering, etc.),

- `XLABackend.ipynb` (how XLA actually compiles and fuses a jitted program), and 

- `StableHLOExport.ipynb` (inspecting compiled cost/memory, and exporting a compiled artifact).

## We would like your feedback

Thank you very much for attending this session!
We hope this experience was both useful and interesting for you.

Please let us know what we can improve.